In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
WORK_DIR="/home/magnolia/DataScience/Stellar_Class"
TRAIN_PATH=Path(WORK_DIR,"data/train.csv")
TEST_PATH=Path(WORK_DIR, "data/test.csv")
MODEL_DIR=Path(WORK_DIR,"models")

df=pd.read_csv(TRAIN_PATH).set_index('id')
target=df['class']
X=df.drop(['class'], axis=1)
# map targets to numericals 

target_map={"GALAXY":0, "QSO": 1, "STAR": 2}
rev_map={0:"GALAXY", 1:"QSO", 2:"STAR"}

target=target.map(target_map)

In [92]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import PolynomialFeatures

In [93]:
X_train, X_test, y_train, y_test = train_test_split(X, target, stratify=target, random_state=10)

In [94]:
num_cols=X.select_dtypes(['float', 'int']).columns.to_list()
cat_cols=X.select_dtypes('object').columns.to_list()

num_est=Pipeline([('mm', MinMaxScaler()), ('pf', PolynomialFeatures(degree=3))])
cat_est=Pipeline([('encoder', OneHotEncoder())])

pipeline=ColumnTransformer([('num_pipe', num_est, num_cols),
                            ('col_pipe', cat_est, cat_cols)])

In [95]:
X_train=pipeline.fit_transform(X_train)
X_test=pipeline.transform(X_test)

In [7]:
from xgboost import XGBClassifier
from sklearn.metrics import balanced_accuracy_score

model1=XGBClassifier(booster='dart', max_depth=8, learning_rate=0.1)
model1.fit(X_train, y_train)
print(f"Train: {balanced_accuracy_score(y_train, model1.predict(X_train))}\nTest: {balanced_accuracy_score(y_test, model1.predict(X_test))}")

Train: 0.9619170342656287
Test: 0.9531435397843101


In [8]:
feat_imps=model1.feature_importances_
num_trans_feat=pipeline['num_pipe'].get_feature_names_out().tolist()
cat_trans_feat=pipeline['col_pipe'].get_feature_names_out().tolist()
trans_feat=num_trans_feat+cat_trans_feat

In [ ]:
df_feat_imp=pd.DataFrame({'feature': trans_feat, 'importance': feat_imps})
ordered_df_feat_imp=df_feat_imp.sort_values(by='importance')
my_range=range(1, len(df_feat_imp)+1)
plt.stem(ordered_df_feat_imp['importance'])

In [ ]:
df_feat_imp_gt_mean_imp=ordered_df_feat_imp[ordered_df_feat_imp['importance']>ordered_df_feat_imp['importance'].median()]
ordered_df_feat_imp_gt_mean_imp=df_feat_imp_gt_mean_imp.sort_values(by='importance')
my_range=range(len(ordered_df_feat_imp_gt_mean_imp))

plt.stem(ordered_df_feat_imp_gt_mean_imp['importance'], orientation='horizontal')
plt.yticks(my_range, ordered_df_feat_imp_gt_mean_imp['feature'])


In [36]:
# select features with feature importance > median(feature_importance_)
df_feat_imp_gt_median_imp=ordered_df_feat_imp[ordered_df_feat_imp['importance']>ordered_df_feat_imp['importance'].median()]
features_index_=df_feat_imp_gt_mean_imp.index
X_train_feat    = X_train[:,features_index_]
X_test_feat     = X_test[:,features_index_]

In [79]:
# Hyper parameter Optimisation
from sklearn.metrics import balanced_accuracy_score
param_range=np.linspace(0.0001,0.001,5)

def Optimise(X_train, y_train, X_test, y_test, param_range: list):
    train_  = []
    test_   = []

    for param in param_range:
        model           = XGBClassifier(gamma=param)
        model.fit(X_train, y_train)
        y_train_pred    = model.predict(X_train)
        y_test_pred     = model.predict(X_test)
        train_score     = balanced_accuracy_score(y_train, y_train_pred)
        test_score      = balanced_accuracy_score(y_test, y_test_pred)
        train_.append(train_score)
        test_.append(test_score)
    
    df = pd.DataFrame({'param': param_range, 'train': train_, 'test': test_})
    return df

Optimise(X_train_feat, y_train, X_test_feat, y_test, param_range=param_range)

,param,train,test
0,0.000100,0.964037,0.954795
1,0.000325,0.964037,0.954795
2,0.000550,0.964289,0.954926
3,0.000775,0.964289,0.954926
4,0.001000,0.964289,0.954926


In [ ]:
# optimisation -- 'Optimsed' Model
model=XGBClassifier(booster='dart', max_depth=8, learning_rate=0.1, grow_policy='lossguide', gamma=0.001)
model.fit(X_train_feat, y_train)
print(f"Train: {balanced_accuracy_score(y_train, model.predict(X_train_feat))}\nTest: {balanced_accuracy_score(y_test, model.predict(X_test_feat))}")

Train: 0.9617962157738941
Test: 0.9533048725510986


In [96]:
# Fit Model on Entire Dataset 
TRAIN_X=pipeline.transform(X)[:, features_index_]
model=XGBClassifier(booster='dart', max_depth=8, learning_rate=0.1, grow_policy='lossguide', gamma=0.001)
model.fit(TRAIN_X, target)
print(f"Balanced Accuracy Score: {balanced_accuracy_score(target, model.predict(TRAIN_X))}")

Balanced Accuracy Score: 0.9602205271814747


In [97]:
# Prediction
df_pred=pd.read_csv(TEST_PATH).set_index('id')
df_pred_transform=pipeline.transform(df_pred)
df_pred_transform= df_pred_transform[:, features_index_]
y_pred=model.predict(df_pred_transform)
y_pred=[rev_map[p] for p in y_pred]
pd.DataFrame({'id':df_pred.index, 'class':y_pred}).set_index('id').to_csv(Path(WORK_DIR, 'submission', 'submission7.csv'))